# 10.1 深度神经网络架构

### 批量归一化

**$\beta$-平滑 ($\beta$-smoothness)**
$\beta$ 越小，表示函数 f 越平滑
$$\|\nabla f(\mathbf{x}) - \nabla f(\mathbf{y})\|^2 \leq \beta \|\mathbf{x} - \mathbf{y}\|^2$$

$\beta$ 越小，可以使用的 学习率 (Learning Rate) 就越大：
- 即 y 离 x 较远时，梯度方向也不会有太大改变，一次梯度下降不会走偏

**Batch Normalization (BN) 的介入**
- 对深度神经网络失效：即使输入是标准化的，经过几层非线性变换和权重乘法后，中间层的输出（即下一层的输入）也会变得不再标准化（分布偏移）
- 在内部层中引入 BN 层，改善中间（复合函数组分）的平滑度
- 即使网络加深也可以使用更大的学习率，加快收敛速度，但对精度影响不大

**BN 的步骤**
- **Reshape**: input $\boldsymbol{\mathbf{X}}$ into 2D (no change for 2D input $\boldsymbol{\mathbf{X}} \in \mathbb{R}^{n \times p}$)  
  - $\boldsymbol{\mathbf{X}} \in \mathbb{R}^{n \times c \times w \times h} \rightarrow \boldsymbol{\mathbf{X}}' \in \mathbb{R}^{nwh \times c}$(batch$n$, channel $c$, width $w$, height $h$)  
- **Normalize**: by standardization each column(每个特征维) $\boldsymbol{\mathbf{x}}_j'$, $j = 1, \dots, p$
    - $\hat{\boldsymbol{\mathbf{x}}}_j' \leftarrow \frac{\boldsymbol{\mathbf{x}}_j' - \operatorname{mean}(\boldsymbol{\mathbf{x}}_j')}{\operatorname{std}(\boldsymbol{\mathbf{x}}_j')}$
- **Recovery**$: \boldsymbol{\mathbf{Y}}'$ with $\boldsymbol{\mathbf{y}}_j' = \gamma_j \hat{\boldsymbol{\mathbf{x}}}_j + \beta_j$ as the $j$-th column, $\gamma_j, \beta_j$ are parameters  
- Output $\boldsymbol{\mathbf{Y}}$ by reshaping $\boldsymbol{\mathbf{Y}}'$ to the same shape as $\boldsymbol{\mathbf{X}}$

**局限性**：维护的批量内均值和方差不稳定时 BN 不适用，如 GAN、RNN、Adversarial Attack中

In [ ]:
def batch_norm(X, gamma, beta, moving_mean, moving_var, eps, momentum):
    if not torch.is_grad_enabled():  # In prediction mode
        # 不能现场计算均值和方差，使用训练时记录的全局均值和方差
        X_hat = (X - moving_mean) / torch.sqrt(moving_var + eps) # 避免除零
    else:
        assert len(X.shape) in (2, 4)
        if len(X.shape) == 2: 
            mean = X.mean(dim=0) # (n, f) -> (1, f) 消灭批量维，每个特征维生成一个均值
            var = ((X - mean)**2).mean(dim=0)
        else:
            # (n, c, w, h) -> (1, c, 1, 1) 将通道视为特征，不同像素都认为是不同样本
            # 并没有真的 reshape
            mean = X.mean(dim=(0, 2, 3), keepdim=True)
            var = ((X - mean)**2).mean(dim=(0, 2, 3), keepdim=True)
        X_hat = (X - mean) / torch.sqrt(var + eps)
        # 平滑移动均值和方差 （真正的全局均值和方差实现起来比较麻烦）
        moving_mean = momentum * moving_mean + (1.0 - momentum) * mean 
        moving_var = momentum * moving_var + (1.0 - momentum) * var
    Y = gamma * X_hat + beta
    return Y, moving_mean, moving_var

### 层归一化
对于 RNN ($n \times p \times t$ 变长输入)，不能让 BN 对不同的时间步使用同一套均值和方差（抖动比较大）

- If apply to RNN, BN needs to maintain separated moving statistics for each time step  
  - Problematic for very long sequences during inference  
- Layer normalization reshapes input $\mathbf{X} \in \mathbb{R}^{n \times p} \to \mathbf{X}' \in \mathbb{R}^{p \times n}$ or $\mathbf{X} \in \mathbb{R}^{n \times c \times w \times h} \to \mathbf{X}' \in \mathbb{R}^{cwh \times n}$, rest is same with BN （对每个样本，按行算均值和方差）
  - Normalizing within each example, up to current time step  
  - Consistent between training and inference  
  - **Popularized by Transformers**

***More Normalizations***
- Modify "reshape", e.g.  
  - InstanceNorm: $n \times c \times w \times h \rightarrow wh \times cn$- GroupNorm:$n \times c \times w \times h \rightarrow swh \times gn$with$c = sg$- CrossNorm: swap mean/std between a pair of features  
- Modify "normalize": e.g. whitening（将协方差矩阵变为单位矩阵，降低特征间相关性）
- Modify "recovery": e.g. replace$\gamma, \beta$ with a dense layer  
- Apply to weights or gradients